# Final_df_formatting

In [1]:
# load libraries
import os
import sys
import numpy as np
import pandas as pd
from datetime import datetime
from itertools import permutations, combinations, product
import itertools
DATE = datetime.now().strftime("%Y-%m-%d")

import matplotlib.pyplot as plt
#import seaborn as sns
from matplotlib.ticker import FuncFormatter
import pickle
import math
from upsetplot import plot, UpSet
from upsetplot import generate_counts, plot

In [38]:
data=pd.read_csv('/Users/graceramey/Library/CloudStorage/Box-Box/capra_lab/projects/dominant_negative_disease_editing/results/dnd_593_2026_02_19.csv')
data.columns

Index(['hgnc_symbol', 'GENE ID (HGNC)', 'DISEASE LABEL', 'DISEASE ID (MONDO)',
       'MOI', 'CLASSIFICATION', 'HI Score', '%HI', 'pLI', 'LOEUF', 'total_plp',
       'missense_plp', 'nonsense_plp', 'ensg', 'chrom', 'obs_lof', 'exp_lof',
       'prior_mean', 'post_mean', 'post_lower_95', 'post_upper_95',
       'indel_targetable_pre_NMD_assessment', 'num_indel_vars',
       'crisproff_targetable', 'num_crisproff_vars', 'base_editable',
       'num_base_editable_vars', 'excision_targetable', 'num_excision_vars',
       'num_excision_pairs', 'HPO_term_list', 'Approved name',
       'indel_pam_targetable', 'crisproff_pam_targetable',
       'base_edit_pam_targetable', 'excision_pam_targetable',
       'hets_across_strats', 'prop_hets_across_strats', 'num_indel_hets',
       'indel_hets_prop', 'num_crisproff_hets', 'crisproff_hets_prop',
       'num_base_edit_hets', 'base_edit_hets_prop', 'num_excision_hets',
       'excision_hets_prop', 'hets_across_strats_prePAM',
       'prop_hets_across

In [39]:
# change column ordering
data = data[['hgnc_symbol', 'GENE ID (HGNC)', 'DISEASE LABEL', 'DISEASE ID (MONDO)',
       'MOI', 'CLASSIFICATION', 'HI Score', '%HI', 'pLI', 'LOEUF', 'ensg', 'chrom', 'obs_lof', 'exp_lof',
       'prior_mean', 'post_mean', 'post_lower_95', 'post_upper_95', 'indel_targetable_post_NMD_assessment', 
             'num_vars_inducing_NMD',
             'crisproff_targetable', 'num_crisproff_vars', 'base_editable',
       'num_base_editable_vars', 'excision_targetable', 'num_excision_vars',
       'num_excision_pairs',
        'hets_across_strats_prePAM',
       'prop_hets_across_strats_prePAM', 'num_indel_hets_prePAM',
       'indel_hets_prop_prePAM', 'num_crisproff_hets_prePAM',
       'crisproff_hets_prop_prePAM', 'num_base_edit_hets_prePAM',
       'base_edit_hets_prop_prePAM', 'num_excision_hets_prePAM',
       'excision_hets_prop_prePAM',
       'indel_pam_targetable', 'crisproff_pam_targetable',
       'base_edit_pam_targetable', 'excision_pam_targetable',
       'hets_across_strats', 'prop_hets_across_strats', 'num_indel_hets',
       'indel_hets_prop', 'num_crisproff_hets', 'crisproff_hets_prop',
       'num_base_edit_hets', 'base_edit_hets_prop', 'num_excision_hets',
       'excision_hets_prop',
       'indels_num_haps_four_guides_alg2', 'indels_num_hets_four_guides_alg2',
       'indels_num_haps_all_guides_alg2', 'indels_num_hets_all_guides_alg2',
       'indels_max_guides', 'CRISPRoff_num_haps_four_guides_alg2',
       'CRISPRoff_num_hets_four_guides_alg2',
       'CRISPRoff_num_haps_all_guides_alg2',
       'CRISPRoff_num_hets_all_guides_alg2', 'CRISPRoff_max_guides',
        'combined_base_edit_num_haps_four_guides_alg2',
       'combined_base_edit_num_hets_four_guides_alg2',
       'combined_base_edit_num_haps_all_guides_alg2',
       'combined_base_edit_num_hets_all_guides_alg2',
       'combined_base_edit_max_guides',
       'excision_num_haps_four_guides_alg2',
       'excision_num_hets_four_guides_alg2',
       'excision_num_haps_all_guides_alg2',
       'excision_num_hets_all_guides_alg2', 'excision_max_guides'
       ]]

In [40]:
# rename some columns
data=data.copy()
data.rename(columns={
    'post_mean':'s_het', 
    'post_lower_95':'s_het_lower_95', 
    'post_upper_95': 's_het_upper_95',
    'indel_targetable_post_NMD_assessment':'exon_disr_targetable', 
    'num_vars_inducing_NMD':'num_exon_disr_vars',
    'crisproff_targetable':'epi_sil_targetable', 
    'num_crisproff_vars':'num_epi_sil_vars', 
    'base_editable':'ss_disr_targetable',
    'num_base_editable_vars':'num_ss_disr_vars', 
    'indel_pam_targetable':'exon_disr_spcas9_targetable', 
    'crisproff_pam_targetable':'epi_sil_spcas9_targetable',
    'base_edit_pam_targetable':'ss_disr_spcas9_targetable', 
    'excision_pam_targetable':'excision_spcas9_targetable',
    'hets_across_strats':'num_hets_all_strats', 
    'prop_hets_across_strats':'prop_1KG_hets_all_strats', 
    'num_indel_hets':'num_hets_exon_disr',
    'indel_hets_prop':'prop_1KG_hets_exon_disr', 
    'num_crisproff_hets':'num_hets_epi_sil', 
    'crisproff_hets_prop':'prop_1KG_hets_epi_sil',
    'num_base_edit_hets':'num_hets_ss_disr', 
    'base_edit_hets_prop':'prop_1KG_hets_ss_disr', 
    'num_excision_hets':'num_hets_excision',
    'excision_hets_prop':'prop_1KG_hets_excision',
    'hets_across_strats_prePAM':'num_hets_preCas_all_strats',
    'prop_hets_across_strats_prePAM':'prop_hets_preCas_all_strats', 
    'num_indel_hets_prePAM':'num_hets_preCas_exon_disr',
    'indel_hets_prop_prePAM':'prop_hets_preCas_exon_disr', 
    'num_crisproff_hets_prePAM':'num_hets_preCas_epi_sil',
    'crisproff_hets_prop_prePAM':'prop_hets_preCas_epi_sil',
    'num_base_edit_hets_prePAM':'num_hets_preCas_ss_disr',
    'base_edit_hets_prop_prePAM':'prop_hets_preCas_ss_disr',
    'num_excision_hets_prePAM':'num_hets_preCas_excision',
    'excision_hets_prop_prePAM':'prop_hets_preCas_excision',
    'indels_num_haps_four_guides_alg2':'num_haplos_targeted_4g_exon_disr', 
    'indels_num_hets_four_guides_alg2':'num_hets_targeted_4g_exon_disr',
    'indels_num_haps_all_guides_alg2':'num_haplos_targeted_ag_exon_disr', 
    'indels_num_hets_all_guides_alg2':'num_hets_targeted_ag_exon_disr',
    'indels_max_guides':'max_num_guides_exon_disr',
    'CRISPRoff_num_haps_four_guides_alg2':'num_haplos_targeted_4g_epi_sil',
    'CRISPRoff_num_hets_four_guides_alg2':'num_hets_targeted_4g_epi_sil',
    'CRISPRoff_num_haps_all_guides_alg2':'num_haplos_targeted_ag_epi_sil',
    'CRISPRoff_num_hets_all_guides_alg2':'num_hets_targeted_ag_epi_sil', 
    'CRISPRoff_max_guides':'max_num_guides_epi_sil',
    'combined_base_edit_num_haps_four_guides_alg2':'num_haplos_targeted_4g_ss_disr',
    'combined_base_edit_num_hets_four_guides_alg2':'num_hets_targeted_4g_ss_disr',
    'combined_base_edit_num_haps_all_guides_alg2':'num_haplos_targeted_ag_ss_disr',
    'combined_base_edit_num_hets_all_guides_alg2':'num_hets_targeted_ag_ss_disr',
    'combined_base_edit_max_guides':'max_num_guides_ss_disr',
    'excision_num_haps_four_guides_alg2':'num_haplos_targeted_4g_excision',
    'excision_num_hets_four_guides_alg2':'num_hets_targeted_4g_excision',
    'excision_num_haps_all_guides_alg2':'num_haplos_targeted_ag_excision',
    'excision_num_hets_all_guides_alg2':'num_hets_targeted_ag_excision', 
    'excision_max_guides':'max_num_guides_excision'},inplace=True)
data

,hgnc_symbol,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,CLASSIFICATION,HI Score,%HI,pLI,LOEUF,...,num_haplos_targeted_4g_ss_disr,num_hets_targeted_4g_ss_disr,num_haplos_targeted_ag_ss_disr,num_hets_targeted_ag_ss_disr,max_num_guides_ss_disr,num_haplos_targeted_4g_excision,num_hets_targeted_4g_excision,num_haplos_targeted_ag_excision,num_hets_targeted_ag_excision,max_num_guides_excision
0,AARS1,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,Definitive,NoEvidence,24.74,0.000000e+00,0.77,...,994.0,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0
1,AARS1,HGNC:20,AARS1-related leukoencephalopathy,MONDO:1010132,AD,Limited,NoEvidence,24.74,0.000000e+00,0.77,...,994.0,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0
2,ABCB6,HGNC:47,"microphthalmia, isolated, with coloboma 7",MONDO:0013783,AD,Limited,NaN,NaN,2.001700e-25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABCB6,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,Moderate,NaN,NaN,2.001700e-25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABCC6,HGNC:57,inherited pseudoxanthoma elasticum,MONDO:0100091,SD,Definitive,NaN,NaN,1.235100e-33,NaN,...,NaN,NaN,NaN,NaN,NaN,1140.0,570.0,1402.0,701.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732,ZIC1,HGNC:12872,craniosynostosis 6,MONDO:0014705,AD,Definitive,NoEvidence,7.94,1.000000e+00,0.31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
733,ZMIZ1,HGNC:16493,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,NaN,NaN,1.000000e+00,NaN,...,2336.0,1168.0,2336.0,1168.0,2.0,2412.0,1206.0,5080.0,2540.0,50.0
734,ZMYND8,HGNC:9397,syndromic complex neurodevelopmental disorder,MONDO:0800439,AD,Strong,NaN,NaN,1.000000e+00,NaN,...,NaN,NaN,NaN,NaN,NaN,1004.0,502.0,4087.0,2042.0,57.0
735,ZNF292,HGNC:18410,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,LittleEvidence,26.56,1.000000e+00,0.16,...,NaN,NaN,NaN,NaN,NaN,2398.0,1199.0,4064.0,2032.0,60.0


In [41]:
# ensure proper classification level
data = data[data['CLASSIFICATION'].isin(['Strong','Definitive','Moderate'])]
data

,hgnc_symbol,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,CLASSIFICATION,HI Score,%HI,pLI,LOEUF,...,num_haplos_targeted_4g_ss_disr,num_hets_targeted_4g_ss_disr,num_haplos_targeted_ag_ss_disr,num_hets_targeted_ag_ss_disr,max_num_guides_ss_disr,num_haplos_targeted_4g_excision,num_hets_targeted_4g_excision,num_haplos_targeted_ag_excision,num_hets_targeted_ag_excision,max_num_guides_excision
0,AARS1,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,Definitive,NoEvidence,24.74,0.000000e+00,0.77,...,994.0,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0
3,ABCB6,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,Moderate,NaN,NaN,2.001700e-25,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABCC6,HGNC:57,inherited pseudoxanthoma elasticum,MONDO:0100091,SD,Definitive,NaN,NaN,1.235100e-33,NaN,...,NaN,NaN,NaN,NaN,NaN,1140.0,570.0,1402.0,701.0,20.0
5,ABCC8,HGNC:59,monogenic diabetes,MONDO:0015967,SD,Definitive,AutosomalRecessive,1.79,0.000000e+00,0.83,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
6,ABCC8,HGNC:59,pulmonary arterial hypertension,MONDO:0015924,AD,Moderate,AutosomalRecessive,1.79,0.000000e+00,0.83,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732,ZIC1,HGNC:12872,craniosynostosis 6,MONDO:0014705,AD,Definitive,NoEvidence,7.94,1.000000e+00,0.31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
733,ZMIZ1,HGNC:16493,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,NaN,NaN,1.000000e+00,NaN,...,2336.0,1168.0,2336.0,1168.0,2.0,2412.0,1206.0,5080.0,2540.0,50.0
734,ZMYND8,HGNC:9397,syndromic complex neurodevelopmental disorder,MONDO:0800439,AD,Strong,NaN,NaN,1.000000e+00,NaN,...,NaN,NaN,NaN,NaN,NaN,1004.0,502.0,4087.0,2042.0,57.0
735,ZNF292,HGNC:18410,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,LittleEvidence,26.56,1.000000e+00,0.16,...,NaN,NaN,NaN,NaN,NaN,2398.0,1199.0,4064.0,2032.0,60.0


In [42]:
# add group 1/2 classification
data.insert(1, 'Gene_Group', np.where(data['s_het'] < 0.1, 'Group 2', 'Group 1'))
data

,hgnc_symbol,Gene_Group,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,CLASSIFICATION,HI Score,%HI,pLI,...,num_haplos_targeted_4g_ss_disr,num_hets_targeted_4g_ss_disr,num_haplos_targeted_ag_ss_disr,num_hets_targeted_ag_ss_disr,max_num_guides_ss_disr,num_haplos_targeted_4g_excision,num_hets_targeted_4g_excision,num_haplos_targeted_ag_excision,num_hets_targeted_ag_excision,max_num_guides_excision
0,AARS1,Group 2,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,Definitive,NoEvidence,24.74,0.000000e+00,...,994.0,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0
3,ABCB6,Group 2,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,Moderate,NaN,NaN,2.001700e-25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABCC6,Group 2,HGNC:57,inherited pseudoxanthoma elasticum,MONDO:0100091,SD,Definitive,NaN,NaN,1.235100e-33,...,NaN,NaN,NaN,NaN,NaN,1140.0,570.0,1402.0,701.0,20.0
5,ABCC8,Group 2,HGNC:59,monogenic diabetes,MONDO:0015967,SD,Definitive,AutosomalRecessive,1.79,0.000000e+00,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
6,ABCC8,Group 2,HGNC:59,pulmonary arterial hypertension,MONDO:0015924,AD,Moderate,AutosomalRecessive,1.79,0.000000e+00,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732,ZIC1,Group 1,HGNC:12872,craniosynostosis 6,MONDO:0014705,AD,Definitive,NoEvidence,7.94,1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
733,ZMIZ1,Group 1,HGNC:16493,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,NaN,NaN,1.000000e+00,...,2336.0,1168.0,2336.0,1168.0,2.0,2412.0,1206.0,5080.0,2540.0,50.0
734,ZMYND8,Group 1,HGNC:9397,syndromic complex neurodevelopmental disorder,MONDO:0800439,AD,Strong,NaN,NaN,1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,1004.0,502.0,4087.0,2042.0,57.0
735,ZNF292,Group 1,HGNC:18410,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,LittleEvidence,26.56,1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,2398.0,1199.0,4064.0,2032.0,60.0


In [43]:
# Add HPO term information
path2hpo='/Users/graceramey/Library/CloudStorage/Box-Box/capra_lab/projects/dominant_negative_disease_editing/results/hpo_terms/dnd2organ_system.pkl'
with open(path2hpo,'rb') as fp:
    gene2hpo=pickle.load(fp)
data.insert(7,'HPO_terms',data['hgnc_symbol'].map(gene2hpo))
data

,hgnc_symbol,Gene_Group,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,CLASSIFICATION,HPO_terms,HI Score,%HI,...,num_haplos_targeted_4g_ss_disr,num_hets_targeted_4g_ss_disr,num_haplos_targeted_ag_ss_disr,num_hets_targeted_ag_ss_disr,max_num_guides_ss_disr,num_haplos_targeted_4g_excision,num_hets_targeted_4g_excision,num_haplos_targeted_ag_excision,num_hets_targeted_ag_excision,max_num_guides_excision
0,AARS1,Group 2,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,Definitive,"{Abnormality of the eye, Abnormality of the re...",NoEvidence,24.74,...,994.0,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0
3,ABCB6,Group 2,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,Moderate,"{Abnormality of the eye, Abnormality of the mu...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABCC6,Group 2,HGNC:57,inherited pseudoxanthoma elasticum,MONDO:0100091,SD,Definitive,"{Abnormality of the eye, Abnormality of head o...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1140.0,570.0,1402.0,701.0,20.0
5,ABCC8,Group 2,HGNC:59,monogenic diabetes,MONDO:0015967,SD,Definitive,"{Abnormality of the eye, Abnormality of the en...",AutosomalRecessive,1.79,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
6,ABCC8,Group 2,HGNC:59,pulmonary arterial hypertension,MONDO:0015924,AD,Moderate,"{Abnormality of the eye, Abnormality of the en...",AutosomalRecessive,1.79,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732,ZIC1,Group 1,HGNC:12872,craniosynostosis 6,MONDO:0014705,AD,Definitive,"{Abnormality of the eye, Abnormality of head o...",NoEvidence,7.94,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
733,ZMIZ1,Group 1,HGNC:16493,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,"{Abnormality of the eye, Abnormality of head o...",NaN,NaN,...,2336.0,1168.0,2336.0,1168.0,2.0,2412.0,1206.0,5080.0,2540.0,50.0
734,ZMYND8,Group 1,HGNC:9397,syndromic complex neurodevelopmental disorder,MONDO:0800439,AD,Strong,"{Abnormality of head or neck, Abnormality of t...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1004.0,502.0,4087.0,2042.0,57.0
735,ZNF292,Group 1,HGNC:18410,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,"{Abnormality of the eye, Abnormality of head o...",LittleEvidence,26.56,...,NaN,NaN,NaN,NaN,NaN,2398.0,1199.0,4064.0,2032.0,60.0


In [44]:
# add in pathogenic mutation counts
dom_clean=pd.read_csv('/Users/graceramey/Desktop/UCSF/CapraLab/Conklin_Collabs/Data/GeneSets/dHS/2025-11-13/ClinVar_checkpoints/ClinVar_Omim_dom_annot_reduced_cols.csv')
dom_clean.rename(columns={'gene':'hgnc_symbol'},inplace=True)
data=data.merge(dom_clean, on='hgnc_symbol')
col = data.pop('dominant_mutation_count')  # remove the column
data.insert(7, 'dominant_mutation_count', col)
data

,hgnc_symbol,Gene_Group,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,CLASSIFICATION,dominant_mutation_count,HPO_terms,HI Score,...,num_hets_targeted_4g_ss_disr,num_haplos_targeted_ag_ss_disr,num_hets_targeted_ag_ss_disr,max_num_guides_ss_disr,num_haplos_targeted_4g_excision,num_hets_targeted_4g_excision,num_haplos_targeted_ag_excision,num_hets_targeted_ag_excision,max_num_guides_excision,counter
0,AARS1,Group 2,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,Definitive,10,"{Abnormality of the eye, Abnormality of the re...",NoEvidence,...,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0,0
1,ABCB6,Group 2,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,Moderate,8,"{Abnormality of the eye, Abnormality of the mu...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,ABCC6,Group 2,HGNC:57,inherited pseudoxanthoma elasticum,MONDO:0100091,SD,Definitive,74,"{Abnormality of the eye, Abnormality of head o...",NaN,...,NaN,NaN,NaN,NaN,1140.0,570.0,1402.0,701.0,20.0,2
3,ABCC8,Group 2,HGNC:59,monogenic diabetes,MONDO:0015967,SD,Definitive,272,"{Abnormality of the eye, Abnormality of the en...",AutosomalRecessive,...,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0,3
4,ABCC8,Group 2,HGNC:59,pulmonary arterial hypertension,MONDO:0015924,AD,Moderate,272,"{Abnormality of the eye, Abnormality of the en...",AutosomalRecessive,...,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,ZIC1,Group 1,HGNC:12872,craniosynostosis 6,MONDO:0014705,AD,Definitive,5,"{Abnormality of the eye, Abnormality of head o...",NoEvidence,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,588
656,ZMIZ1,Group 1,HGNC:16493,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,25,"{Abnormality of the eye, Abnormality of head o...",NaN,...,1168.0,2336.0,1168.0,2.0,2412.0,1206.0,5080.0,2540.0,50.0,589
657,ZMYND8,Group 1,HGNC:9397,syndromic complex neurodevelopmental disorder,MONDO:0800439,AD,Strong,1,"{Abnormality of head or neck, Abnormality of t...",NaN,...,NaN,NaN,NaN,NaN,1004.0,502.0,4087.0,2042.0,57.0,590
658,ZNF292,Group 1,HGNC:18410,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,47,"{Abnormality of the eye, Abnormality of head o...",LittleEvidence,...,NaN,NaN,NaN,NaN,2398.0,1199.0,4064.0,2032.0,60.0,591


In [46]:
data.drop(labels=['counter'],axis=1,inplace=True)
data

,hgnc_symbol,Gene_Group,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,CLASSIFICATION,dominant_mutation_count,HPO_terms,HI Score,...,num_haplos_targeted_4g_ss_disr,num_hets_targeted_4g_ss_disr,num_haplos_targeted_ag_ss_disr,num_hets_targeted_ag_ss_disr,max_num_guides_ss_disr,num_haplos_targeted_4g_excision,num_hets_targeted_4g_excision,num_haplos_targeted_ag_excision,num_hets_targeted_ag_excision,max_num_guides_excision
0,AARS1,Group 2,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,Definitive,10,"{Abnormality of the eye, Abnormality of the re...",NoEvidence,...,994.0,497.0,994.0,497.0,2.0,2152.0,1076.0,3142.0,1571.0,24.0
1,ABCB6,Group 2,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,Moderate,8,"{Abnormality of the eye, Abnormality of the mu...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABCC6,Group 2,HGNC:57,inherited pseudoxanthoma elasticum,MONDO:0100091,SD,Definitive,74,"{Abnormality of the eye, Abnormality of head o...",NaN,...,NaN,NaN,NaN,NaN,NaN,1140.0,570.0,1402.0,701.0,20.0
3,ABCC8,Group 2,HGNC:59,monogenic diabetes,MONDO:0015967,SD,Definitive,272,"{Abnormality of the eye, Abnormality of the en...",AutosomalRecessive,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
4,ABCC8,Group 2,HGNC:59,pulmonary arterial hypertension,MONDO:0015924,AD,Moderate,272,"{Abnormality of the eye, Abnormality of the en...",AutosomalRecessive,...,1838.0,919.0,1838.0,919.0,2.0,1774.0,887.0,4101.0,2050.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,ZIC1,Group 1,HGNC:12872,craniosynostosis 6,MONDO:0014705,AD,Definitive,5,"{Abnormality of the eye, Abnormality of head o...",NoEvidence,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
656,ZMIZ1,Group 1,HGNC:16493,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,25,"{Abnormality of the eye, Abnormality of head o...",NaN,...,2336.0,1168.0,2336.0,1168.0,2.0,2412.0,1206.0,5080.0,2540.0,50.0
657,ZMYND8,Group 1,HGNC:9397,syndromic complex neurodevelopmental disorder,MONDO:0800439,AD,Strong,1,"{Abnormality of head or neck, Abnormality of t...",NaN,...,NaN,NaN,NaN,NaN,NaN,1004.0,502.0,4087.0,2042.0,57.0
658,ZNF292,Group 1,HGNC:18410,complex neurodevelopmental disorder,MONDO:0100038,AD,Definitive,47,"{Abnormality of the eye, Abnormality of head o...",LittleEvidence,...,NaN,NaN,NaN,NaN,NaN,2398.0,1199.0,4064.0,2032.0,60.0


In [47]:
# fill some columns with Nan or no value
data.columns

Index(['hgnc_symbol', 'Gene_Group', 'GENE ID (HGNC)', 'DISEASE LABEL',
       'DISEASE ID (MONDO)', 'MOI', 'CLASSIFICATION',
       'dominant_mutation_count', 'HPO_terms', 'HI Score', '%HI', 'pLI',
       'LOEUF', 'ensg', 'chrom', 'obs_lof', 'exp_lof', 'prior_mean', 's_het',
       's_het_lower_95', 's_het_upper_95', 'exon_disr_targetable',
       'num_exon_disr_vars', 'epi_sil_targetable', 'num_epi_sil_vars',
       'ss_disr_targetable', 'num_ss_disr_vars', 'excision_targetable',
       'num_excision_vars', 'num_excision_pairs', 'num_hets_preCas_all_strats',
       'prop_hets_preCas_all_strats', 'num_hets_preCas_exon_disr',
       'prop_hets_preCas_exon_disr', 'num_hets_preCas_epi_sil',
       'prop_hets_preCas_epi_sil', 'num_hets_preCas_ss_disr',
       'prop_hets_preCas_ss_disr', 'num_hets_preCas_excision',
       'prop_hets_preCas_excision', 'exon_disr_spcas9_targetable',
       'epi_sil_spcas9_targetable', 'ss_disr_spcas9_targetable',
       'excision_spcas9_targetable', 'num_hets

In [49]:
cols = ['exon_disr_targetable','epi_sil_targetable','ss_disr_targetable','excision_targetable']

# Option 1: assign back
data[cols] = data[cols].fillna(0)
data['exon_disr_targetable']

0      1.0
1      0.0
2      0.0
3      1.0
4      1.0
      ... 
655    0.0
656    1.0
657    0.0
658    0.0
659    0.0
Name: exon_disr_targetable, Length: 660, dtype: float64

In [51]:
cols = ['exon_disr_spcas9_targetable','epi_sil_spcas9_targetable', 'ss_disr_spcas9_targetable',
       'excision_spcas9_targetable']

# Option 1: assign back
data[cols] = data[cols].fillna(0)
data['exon_disr_spcas9_targetable']

0      1
1      0
2      0
3      1
4      1
      ..
655    0
656    1
657    0
658    0
659    0
Name: exon_disr_spcas9_targetable, Length: 660, dtype: int64

In [54]:
# finally, save the df
data.to_csv('/Users/graceramey/Library/CloudStorage/Box-Box/capra_lab/projects/dominant_negative_disease_editing/results/cleaned_master_df_2025_04_06.csv')